In [108]:
#imports 
import pandas as pd 
import numpy as np 
from pathlib import Path 

DATA_PATH = Path("..")/"data"

raw = DATA_PATH/"raw"/"train.csv" 

df = pd.read_csv(raw, index_col=0)

In [109]:
df.head()

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [110]:
df.shape

(5847, 13)

In [111]:
df.info()

<class 'pandas.DataFrame'>
Index: 5847 entries, 1 to 6018
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               5847 non-null   str    
 1   Location           5847 non-null   str    
 2   Year               5847 non-null   int64  
 3   Kilometers_Driven  5847 non-null   int64  
 4   Fuel_Type          5847 non-null   str    
 5   Transmission       5847 non-null   str    
 6   Owner_Type         5847 non-null   str    
 7   Mileage            5845 non-null   str    
 8   Engine             5811 non-null   str    
 9   Power              5811 non-null   str    
 10  Seats              5809 non-null   float64
 11  New_Price          815 non-null    str    
 12  Price              5847 non-null   float64
dtypes: float64(2), int64(2), str(9)
memory usage: 639.5 KB


Part A - Find Missing Values 
- though about dropping  the new price column becasue it is missing more than 80% of the values, it does not make snese to use the remaining 20% and impute with but since part b is looking for new price col i can't drop it so will impute with median 
- The other columns with the missing values such as mileage, seats, power, engine will be imputed. 
        - Seats will be mode (can't take decimal seats)
        - milage power and engine will be median 

In [112]:
missing_count = df.isna().sum()
print("These are the missimg values before imputation \n",missing_count)
#Imputation will required numeric valeus 

These are the missimg values before imputation 
 Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 36
Power                  36
Seats                  38
New_Price            5032
Price                   0
dtype: int64


In [113]:
missing_percent = (df.isna().sum()/len(df))*100

print ("The missing percent for each column is\n", missing_percent)

The missing percent for each column is
 Name                  0.000000
Location              0.000000
Year                  0.000000
Kilometers_Driven     0.000000
Fuel_Type             0.000000
Transmission          0.000000
Owner_Type            0.000000
Mileage               0.034206
Engine                0.615700
Power                 0.615700
Seats                 0.649906
New_Price            86.061228
Price                 0.000000
dtype: float64


In [114]:
#seats = number 
df["Seats"] = df["Seats"].fillna(df["Seats"].mode()[0])
print("Number of rows with missing vlaues in Seats Column:",df["Seats"].isna().sum())

Number of rows with missing vlaues in Seats Column: 0


Part B - Remove Units

In [115]:

#https://pandas.pydata.org/docs/getting_started/intro_tutorials/10_text_data.html
cols = ["Mileage", "Engine", "Power", "New_Price"]

for col in cols: 
    df[col] = pd.to_numeric(df[col].str.split().str[0],errors = "coerce")



In [116]:
print(df["Mileage"].dtype)
print(df["Engine"].dtype)
print(df["Power"].dtype)
print(df["New_Price"].dtype)

float64
float64
float64
float64


In [117]:
# countinuing part A 

for col in cols: 
    df[col] = df[col].fillna(df[col].median())

print(df.isna().sum())

Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
New_Price            0
Price                0
dtype: int64


Part C - One Hot Encoding 

In [118]:
#https://www.geeksforgeeks.org/python/how-to-convert-categorical-variable-to-numeric-in-pandas/
df = pd.get_dummies(df,columns=["Fuel_Type","Transmission"], dtype=int)
df.head()

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582.0,126.20,5.0,11.48,12.50,1,0,0,0,1
2,Honda Jazz V,Chennai,2011,46000,First,13.00,1199.0,88.70,5.0,8.61,4.50,0,0,1,0,1
3,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248.0,88.76,7.0,11.48,6.00,1,0,0,0,1
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968.0,140.80,5.0,11.48,17.74,1,0,0,1,0
6,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461.0,63.10,5.0,11.48,3.50,1,0,0,0,1


Part D - New Feature

In [119]:
# price increase 
df["Price_Change"] = df["New_Price"] - df["Price"]
df.head()

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Price_Change
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582.0,126.20,5.0,11.48,12.50,1,0,0,0,1,-1.02
2,Honda Jazz V,Chennai,2011,46000,First,13.00,1199.0,88.70,5.0,8.61,4.50,0,0,1,0,1,4.11
3,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248.0,88.76,7.0,11.48,6.00,1,0,0,0,1,5.48
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968.0,140.80,5.0,11.48,17.74,1,0,0,1,0,-6.26
6,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461.0,63.10,5.0,11.48,3.50,1,0,0,0,1,7.98


Part E - Six Operations 

In [120]:
selected = df[["Name", "Price", "Year", "Location"]]
print("Select Operation")
display(selected.head())

filter = df[df["Mileage"]> 14.35]
print("Filter Operation (Milage > 14.35)")
display(filter.head())

df = df.rename(columns={"Kilometers_Driven":"Km_Driven"})
print("Rename Operation")
renamed_columns = df.columns.tolist()
print(renamed_columns)

df["Price_per_1000km"] = df["Price"] / (df["Km_Driven"]/1000)
print("\nMutate Operation")
display(df[["Price", "Km_Driven", "Price_per_1000km"]].head())

arrange = df.sort_values("Km_Driven", ascending = True)
print("\nArrange Operation")
display(arrange[["Name", "Price", "Km_Driven", "Owner_Type"]].head())

summary = df.groupby(["Owner_Type", "Location"]).agg(
    avg_price = ("Price", "mean"), 
    avg_mileage = ("Mileage","mean"), 
)
print("Summarize Operation")
display(summary)

Select Operation


,Name,Price,Year,Location
1,Hyundai Creta 1.6 CRDi SX Option,12.50,2015,Pune
2,Honda Jazz V,4.50,2011,Chennai
3,Maruti Ertiga VDI,6.00,2012,Chennai
4,Audi A4 New 2.0 TDI Multitronic,17.74,2013,Coimbatore
6,Nissan Micra Diesel XV,3.50,2013,Jaipur


Filter Operation (Milage > 14.35)


,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Price_Change
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582.0,126.20,5.0,11.48,12.50,1,0,0,0,1,-1.02
3,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248.0,88.76,7.0,11.48,6.00,1,0,0,0,1,5.48
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968.0,140.80,5.0,11.48,17.74,1,0,0,1,0,-6.26
6,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461.0,63.10,5.0,11.48,3.50,1,0,0,0,1,7.98
8,Volkswagen Vento Diesel Comfortline,Pune,2013,64430,First,20.54,1598.0,103.60,5.0,11.48,5.20,1,0,0,0,1,6.28


Rename Operation
['Name', 'Location', 'Year', 'Km_Driven', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'New_Price', 'Price', 'Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual', 'Price_Change']

Mutate Operation


,Price,Km_Driven,Price_per_1000km
1,12.50,41000,0.304878
2,4.50,46000,0.097826
3,6.00,87000,0.068966
4,17.74,40670,0.436194
6,3.50,86999,0.040230



Arrange Operation


,Name,Price,Km_Driven,Owner_Type
1361,Maruti Alto 800 2016-2019 VXI,3.60,171,First
5606,Maruti Wagon R ZXI AMT 1.2,6.25,600,First
1198,Volkswagen Polo 1.0 MPI Trendline,5.68,1000,First
2646,Ford Figo 2015-2019 1.2P Sports Edition MT,6.80,1000,First
1598,Tata Xenon XT EX 4X2,7.50,1000,First


Summarize Operation


avg_price  avg_mileage
Owner_Type     Location                          
First          Ahmedabad    8.575055    19.184176
               Bangalore   14.440749    17.337004
               Chennai      9.062883    18.700706
               Coimbatore  15.416678    17.872633
               Delhi        9.865336    17.923812
               Hyderabad   10.339494    18.842591
               Jaipur       6.274899    19.730270
               Kochi       11.282876    18.676975
               Kolkata      5.794970    19.297131
               Mumbai      10.178307    17.565601
               Pune         7.858362    18.434615
Fourth & Above Bangalore    3.250000    17.000000
               Chennai      5.300000    11.250000
               Coimbatore   1.970000    13.800000
               Jaipur       2.350000    16.100000
               Mumbai       4.000000    12.800000
               Pune         2.575000    14.350000
Second         Ahmedabad    8.527778    17.227222
               Bangalore   12.384375    15.760536
               Chennai      6.171638    17.433103
               Coimbatore  12.624921    17.809206
               Delhi        8.779780    16.935824
               Hyderabad    7.185325    17.199870
               Jaipur       5.291828    17.918817
               Kochi       12.362424    17.146970
               Kolkata      4.726667    17.626667
               Mumbai       6.882393    15.893419
               Pune         5.395159    16.752038
Third          Bangalore    6.462500    14.582500
               Chennai      3.348750    16.857500
               Coimbatore  42.910000     8.000000
               Delhi       45.783333    11.993333
               Jaipur       2.506154    16.213077
               Kochi        1.865000    14.650000
               Mumbai       5.632500    14.105000
               Pune         2.927857    16.612857

In [121]:
df.to_csv(DATA_PATH/"clean"/"train_clean.csv")